# MaiAlgo pipeline (CPD + TPS, HaN-Seg subject)
Run the cells top to bottom. Notes: `Document/MaiAlgo/MaiAlgo_Notebook_Guide.md`.

## 0. Config

In [2]:
import os, re, sys, json, time, shutil, warnings
from pathlib import Path
import numpy as np, pandas as pd, pyvista as pv
import scipy.sparse as sp
from scipy.interpolate import RBFInterpolator
from scipy.spatial import cKDTree
from scipy.sparse.linalg import splu
from pycpd import AffineRegistration, DeformableRegistration, RigidRegistration
warnings.filterwarnings('ignore')

DATA_ROOT = os.environ.get('BIODAT_ROOT', r'G:/My Drive/BioDat_Yusuke_lab')   # folder that contains Data/
CASE = 'case_02'            # HaN-Seg case in Data/HaN-Seg/GT/usable
NAME = 'C_nb'               # run name -> Document/MaiAlgo/out_<CASE>_<NAME>

INIT = 'eyes'               # start: 'centroid' | 'eyes'  (anchor both clouds on the eye centroids)
CROP = 'eyes'               # subject crop: 'auto' (shoulder width) | 'eyes' (same distance below eyes as template)
CPD_W = 0.0                 # CPD outlier weight (0 = baseline)
USE_ICP = True              # Stage 6 non-rigid ICP
ARAP_ITERS, ARAP_PROX = 30, 0.0

N_PTS, N_LM = 3000, 2000
CPD_ALPHA, CPD_BETA, CPD_MAX_ITER, CPD_TOL = 2.0, 2.0, 100, 1e-6
TPS_SMOOTHING = 1e-3
ICP_ITERS, ICP_MAX_DIST, ICP_STEP, ICP_K = 10, 15.0, 0.5, 8
AFFINE_SV_CLAMP = (0.8, 1.25)
BLEND_MM, CONF_K, CONF_ITERS = 10.0, 8, 3
AXIS_MAP = np.array([[0, 1, 0], [-1, 0, 0], [0, 0, 1]], float)    # THUMS (X=ant,Y=left,Z=up) -> LPS-like

_here = Path(os.getcwd()).resolve()
REPO = next(p for p in [_here, *_here.parents] if (p / 'Document' / 'MaiAlgo').exists())
MAI = REPO / 'Document' / 'MaiAlgo'
OUT = MAI / f'out_{CASE}_{NAME}'
OUT.mkdir(exist_ok=True)
GT = Path(DATA_ROOT) / 'Data/HaN-Seg/GT/usable' / CASE / 'stl' / CASE
np.random.seed(0)
print(CASE, NAME, '->', OUT)

case_02 C_nb -> C:\Mai\BME\BioDat Work\Yusuke_work\Reconstruction-THUMS-FEA\Document\MaiAlgo\out_case_02_C_nb


## 1. Template (THUMS head)

In [3]:
PART_NAMES = {}
for line in open(Path(DATA_ROOT) / 'Data/Ls_dyna_model/Head_V6.k', errors='ignore'):
    if line.startswith('$HMNAME COMPS'):
        m = re.match(r'\$HMNAME COMPS\s*(\d+)(.*)', line.strip())
        if m: PART_NAMES[int(m.group(1))] = m.group(2).strip()

tmpl = pv.read(str(Path(DATA_ROOT) / 'Data/mesh_model/Head_V6.vtu'))
pids = tmpl.cell_data['part_id']
node_id = np.asarray(tmpl.point_data['node_id'])
id2idx = {int(n): i for i, n in enumerate(node_id)}

SKIN_PIDS = [p for p in range(88000160, 88000241) if '_out_' in PART_NAMES.get(p, '')]
EYE_PIDS = list(range(88000173, 88000217))
skin = tmpl.extract_cells(np.where(np.isin(pids, SKIN_PIDS))[0]).extract_surface(algorithm='dataset_surface').triangulate()
skin_idx = np.array([id2idx[int(n)] for n in skin.point_data['node_id']])

all_pts0 = np.asarray(tmpl.points, float) @ AXIS_MAP.T         # template in the subject frame
skin_pts0 = all_pts0[skin_idx]
eyes_t = np.asarray(tmpl.extract_cells(np.where(np.isin(pids, EYE_PIDS))[0]).points, float) @ AXIS_MAP.T
tmpl_eye_c = eyes_t.mean(0)
print(f'{tmpl.n_points:,} nodes, skin surface {len(skin_idx):,} pts, template eye centroid {tmpl_eye_c.round(1)}')

179,676 nodes, skin surface 6,182 pts, template eye centroid [ -0.   90.6 647.8]


## 2. Subject (HaN-Seg skin)

In [ ]:
skin_full = np.asarray(pv.read(f'{GT}_skin_surface.stl').points, float)
gt_eye_c = np.asarray(pv.read(f'{GT}_eyes_OAR.stl').points, float).mean(0)

if CROP == 'eyes':
    D = tmpl_eye_c[2] - skin_pts0[:, 2].min()          # template skin bottom is D mm below its eyes
    z_cut = gt_eye_c[2] - D
else:
    for z_cut in np.sort(np.unique(np.round(skin_full[:, 2]))):
        keep = skin_full[skin_full[:, 2] >= z_cut]
        if np.abs(keep[:, 0] - np.median(keep[:, 0])).max() <= 110.0: break
skin_crop = skin_full[skin_full[:, 2] >= z_cut]
subject_pts = skin_crop[np.random.choice(len(skin_crop), min(N_PTS, len(skin_crop)), replace=False)]
print(f'{CASE}: skin {len(skin_full):,} -> cropped z >= {z_cut:.1f}: {len(skin_crop):,} pts, sampled {len(subject_pts)}')

## 3. Start alignment + landmarks

In [ ]:
if INIT == 'eyes':
    c_fixed, c_moving = gt_eye_c, tmpl_eye_c
else:
    c_fixed, c_moving = subject_pts.mean(0), skin_pts0.mean(0)
scale = 1.0
subj = subject_pts - c_fixed
tmpl_skin = (skin_pts0 - c_moving) * scale
tmpl_all = (all_pts0 - c_moving) * scale


def fps(points, n, seed=0):
    rng = np.random.default_rng(seed)
    n = min(n, len(points))
    sel = np.empty(n, dtype=np.int64)
    sel[0] = rng.integers(len(points))
    dist = np.linalg.norm(points - points[sel[0]], axis=1)
    for k in range(1, n):
        sel[k] = np.argmax(dist)
        dist = np.minimum(dist, np.linalg.norm(points - points[sel[k]], axis=1))
    return sel


fixed_lm = subj[fps(subj, N_LM)]
moving_lm = tmpl_skin[fps(tmpl_skin, N_LM)]
print('start:', INIT, '| landmarks', fixed_lm.shape, moving_lm.shape)

## 4. CPD (rigid, affine, deformable)

In [ ]:
t0 = time.time()
TY_rigid, (s_r, R_r, t_r) = RigidRegistration(X=fixed_lm, Y=moving_lm, w=CPD_W, max_iterations=CPD_MAX_ITER, tolerance=CPD_TOL).register()
print(f'rigid scale {s_r:.4f}')

_, (B_a, t_a) = AffineRegistration(X=fixed_lm, Y=TY_rigid, w=CPD_W, max_iterations=CPD_MAX_ITER, tolerance=CPD_TOL).register()
U, S, Vt = np.linalg.svd(B_a)
TY_affine = TY_rigid @ (U @ np.diag(np.clip(S, *AFFINE_SV_CLAMP)) @ Vt) + t_a
print('affine SV', S.round(3))

TY_cpd, _ = DeformableRegistration(X=fixed_lm, Y=TY_affine, alpha=CPD_ALPHA, beta=CPD_BETA, w=CPD_W,
                                   max_iterations=CPD_MAX_ITER, tolerance=CPD_TOL).register()
print(f'CPD done in {time.time() - t0:.0f}s')

## 5. TPS, ICP, blend

In [ ]:
def graph_smooth(v, pts, k):
    _, idx = cKDTree(pts).query(pts, k=k + 1)
    return v[idx].mean(axis=1)


def nonrigid_icp(src, tgt, iters, max_dist, step, k):
    pts, tree = src.copy(), cKDTree(tgt)
    for _ in range(iters):
        d, nn = tree.query(pts)
        pull = tgt[nn] - pts
        pull[(d / scale) >= max_dist] = 0.0
        pts = pts + step * graph_smooth(pull, pts, k)
    return pts


def damped_blend(orig, warped, tgt):
    tree = cKDTree(tgt)
    conf = np.clip((tree.query(orig)[0] - tree.query(warped)[0]) / scale / BLEND_MM + 0.5, 0.0, 1.0)
    _, knn = cKDTree(warped).query(warped, k=CONF_K + 1)
    for _ in range(CONF_ITERS): conf = conf[knn].mean(axis=1)
    return orig + conf[:, None] * (warped - orig), conf


tps = RBFInterpolator(moving_lm, TY_cpd, kernel='thin_plate_spline', smoothing=TPS_SMOOTHING)
skin_tps = tps(tmpl_skin)
skin_icp = nonrigid_icp(skin_tps, subj, ICP_ITERS, ICP_MAX_DIST, ICP_STEP, ICP_K) if USE_ICP else skin_tps.copy()
skin_final, conf = damped_blend(tmpl_skin, skin_icp, subj)
print(f'blend: kept {(conf >= 0.999).sum()}, reverted {(conf <= 0.001).sum()} / {len(conf)}')

## 6. Skin check + write mesh

In [ ]:
def skin_agreement(src, tgt):
    a, b = cKDTree(tgt).query(src)[0] / scale, cKDTree(src).query(tgt)[0] / scale
    return dict(mean=a.mean(), median=np.median(a), rms=np.sqrt((a ** 2).mean()), max=a.max(),
                chamfer=a.mean() + b.mean(), hausdorff=max(a.max(), b.max()))


stages = [('0. Center', moving_lm), ('2. Rigid CPD', TY_rigid), ('3. Affine CPD', TY_affine), ('4. Non-rigid CPD', TY_cpd),
          ('5. TPS (dense)', skin_tps), ('6. ICP', skin_icp), ('6.5 Blend (final)', skin_final)]
summary = pd.DataFrame([dict(stage=n, **skin_agreement(p, subj)) for n, p in stages]).round(3)
print(summary.to_string(index=False))
summary.to_csv(OUT / 'validation_summary.csv', index=False)


def to_world(p): return p / scale + c_fixed


# interior: one TPS through the final skin, then the skin nodes themselves are overwritten
tps_final = RBFInterpolator(tmpl_skin, skin_final, kernel='thin_plate_spline', smoothing=TPS_SMOOTHING)
coords_reg = to_world(tps_final(tmpl_all))
coords_reg[skin_idx] = to_world(skin_final)


def umeyama(src, dst):
    mu_s, mu_d = src.mean(0), dst.mean(0)
    U_, S_, Vt_ = np.linalg.svd((dst - mu_d).T @ (src - mu_s) / len(src))
    d = np.diag([1, 1, np.sign(np.linalg.det(U_) * np.linalg.det(Vt_))])
    R = U_ @ d @ Vt_
    s = np.trace(np.diag(S_) @ d) / (src - mu_s).var(0).sum()
    return s, R, mu_d - s * R @ mu_s


s_u, R_u, t_u = umeyama(moving_lm, TY_rigid)
coords_init = to_world(tmpl_all)
coords_sim = to_world(s_u * tmpl_all @ R_u.T + t_u)


def save_run(out, coords):
    out = Path(out); out.mkdir(exist_ok=True)
    m = tmpl.copy(); m.points = coords
    m.save(str(out / 'Head_V6_registered.vtu'))
    np.save(out / 'coords_registered.npy', coords)


save_run(OUT, coords_reg)
np.save(OUT / 'coords_init.npy', coords_init)
np.save(OUT / 'coords_similarity.npy', coords_sim)
json.dump(dict(case=CASE, name=NAME, init=INIT, crop=CROP, w=CPD_W, icp=USE_ICP, rigid_scale=float(s_r), z_cut=float(z_cut),
               final=summary.iloc[-1].to_dict()), open(OUT / 'run_meta.json', 'w'), indent=1, default=float)
print('saved', OUT)

## 7. Interior vs ground truth

In [ ]:
CRANIAL = ('frontal', 'parietal', 'temporal', 'occipital')
_cran = [p for p, n in PART_NAMES.items() if n.startswith(CRANIAL) and 88000001 <= p <= 88000099]
STRUCT = {'skull_outer': ([p for p in _cran if '_external_' in PART_NAMES[p]], 'outer_table_surface'),
          'skull_inner': ([p for p in _cran if '_internal_' in PART_NAMES[p]], 'inner_table_surface'),
          'brain': (list(range(88000100, 88000104)) + list(range(88000120, 88000124)), 'brain'),
          'brainstem': ([88000104, 88000105, 88000124, 88000125], 'brainstem_OAR'),
          'eyes': (EYE_PIDS, 'eyes_OAR')}
GT_PTS = {k: np.asarray(pv.read(f'{GT}_{v[1]}.stl').points, float) for k, v in STRUCT.items()}


def surface_samples(coords, pid_list, n=150000):
    m = tmpl.copy(); m.points = coords
    s = m.extract_cells(np.where(np.isin(pids, pid_list))[0]).extract_surface(algorithm='dataset_surface').triangulate().clean()
    V, F = np.asarray(s.points, float), np.asarray(s.faces).reshape(-1, 4)[:, 1:]
    T = V[F]
    area = 0.5 * np.linalg.norm(np.cross(T[:, 1] - T[:, 0], T[:, 2] - T[:, 0]), axis=1)
    rng = np.random.default_rng(0)
    tri = rng.choice(len(F), n, p=area / area.sum())
    u, v = rng.random(n), rng.random(n)
    f = u + v > 1
    u[f], v[f] = 1 - u[f], 1 - v[f]
    return np.vstack([V, T[tri, 0] + u[:, None] * (T[tri, 1] - T[tri, 0]) + v[:, None] * (T[tri, 2] - T[tri, 0])])


def eval_interior(out):
    out = Path(out)
    states = {'init (axis map only)': np.load(out / 'coords_init.npy'),
              'similarity (rigid CPD)': np.load(out / 'coords_similarity.npy'),
              'baseline registered': np.load(out / 'coords_registered.npy')}
    rows = []
    for sname, coords in states.items():
        for k, (pl, _) in STRUCT.items():
            T, G = surface_samples(coords, pl), GT_PTS[k]
            a, b = cKDTree(G).query(T)[0], cKDTree(T).query(G)[0]
            rows.append(dict(state=sname, structure=k, mean_T2G=a.mean(), mean_G2T=b.mean(), mean_sym=0.5 * (a.mean() + b.mean()),
                             hd95=max(np.percentile(a, 95), np.percentile(b, 95)),
                             centroid_offset=np.linalg.norm(T.mean(0) - G.mean(0)), extent_ratio_z=np.ptp(T[:, 2]) / np.ptp(G[:, 2])))
    df = pd.DataFrame(rows).round(2)
    df.to_csv(out / 'interior_vs_gt.csv', index=False)
    return df


df_int = eval_interior(OUT)
print(df_int[df_int.state != 'init (axis map only)'].to_string(index=False))

## 8. Element Jacobians

In [ ]:
sys.path.insert(0, str(REPO / 'Code' / 'python'))
from k_mesh_qa import parse_k_file

nodes_k, solids_k, _ = parse_k_file(str(Path(DATA_ROOT) / 'Data/Ls_dyna_model/Head_V6.k'))
k_ids = np.array(sorted(nodes_k))
X0 = np.array([nodes_k[n] for n in k_ids], float)
k_idx = np.array([id2idx[int(n)] for n in k_ids])               # k-file node order -> vtu point order
e_ids = np.array(sorted(solids_k))
conn_ids = np.array([solids_k[e][1] for e in e_ids])
conn = np.searchsorted(k_ids, conn_ids)
CORNERS = [(0, (1, 3, 4)), (1, (2, 0, 5)), (2, (3, 1, 6)), (3, (0, 2, 7)), (4, (7, 5, 0)), (5, (4, 6, 1)), (6, (5, 7, 2)), (7, (6, 4, 3))]

valid = np.zeros((len(conn), 8), bool)
for c, (a, b, d) in CORNERS:
    cols, ok = [c, a, b, d], np.ones(len(conn), bool)
    for i in range(4):
        for j in range(i + 1, 4): ok &= conn_ids[:, cols[i]] != conn_ids[:, cols[j]]
    valid[:, c] = ok


def corner_sj(X):
    P = X[conn]
    sj = np.full((len(conn), 8), np.nan)
    for c, (a, b, d) in CORNERS:
        e1, e2, e3 = P[:, a] - P[:, c], P[:, b] - P[:, c], P[:, d] - P[:, c]
        det = np.einsum('ij,ij->i', e1, np.cross(e2, e3))
        nrm = np.linalg.norm(e1, axis=1) * np.linalg.norm(e2, axis=1) * np.linalg.norm(e3, axis=1)
        sj[:, c] = np.where(valid[:, c], det / np.maximum(nrm, 1e-300), np.nan)
    return sj


sj0 = corner_sj(X0)
fail0 = np.nanmin(sj0, axis=1) <= 0


def jacobian_check(out, verbose=True):
    out = Path(out)
    sj1 = corner_sj(np.load(out / 'coords_registered.npy')[k_idx])
    fail1 = np.nanmin(sj1, axis=1) <= 0
    flip = ((sj0 * sj1 < 0) & valid).any(1)
    res = dict(solid_elements=int(len(e_ids)), failing_template=int(fail0.sum()), failing_registered=int(fail1.sum()),
               newly_failing=int((~fail0 & fail1).sum()), fixed_by_warp=int((fail0 & ~fail1).sum()), any_corner_sign_change=int(flip.sum()))
    (out / 'jacobian').mkdir(exist_ok=True)
    json.dump(res, open(out / 'jacobian' / 'jacobian_summary.json', 'w'), indent=1)
    if verbose: print(res)
    return res


jac = jacobian_check(OUT)

## 9. ARAP propagation of the skin into the interior

In [ ]:
HEX_E = np.array([[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]])
QUAD_E = np.array([[0,1],[1,2],[2,3],[3,0]])
edges = np.vstack([c[:, HEX_E if ct == 12 else QUAD_E].reshape(-1, 2) for ct, c in tmpl.cells_dict.items()])
edges = edges[edges[:, 0] != edges[:, 1]]
edges = np.unique(np.sort(edges, axis=1), axis=0)
ei, ej = edges[:, 0], edges[:, 1]
N = len(all_pts0)
_, lab = sp.csgraph.connected_components(sp.coo_matrix((np.ones(len(ei)), (ei, ej)), shape=(N, N)), directed=False)
driven = np.isin(lab, np.unique(lab[skin_idx]))        # nodes connected to the skin (eyes are not)


def arap(P_run, iters=ARAP_ITERS, prox=ARAP_PROX):
    target = P_run[skin_idx]
    H = np.c_[all_pts0[skin_idx], np.ones(len(skin_idx))]
    Rest = np.c_[all_pts0, np.ones(N)] @ np.linalg.lstsq(H, target, rcond=None)[0]     # template after one global affine fit
    L0 = np.linalg.norm(Rest[ei] - Rest[ej], axis=1)
    k = L0 > 1e-9
    i, j, w = ei[k], ej[k], 1.0 / np.maximum(L0[k], 1e-3)
    fixed = np.zeros(N, bool); fixed[skin_idx] = True
    free = np.where(driven & ~fixed)[0]
    Lap = sp.coo_matrix((np.r_[w, w, -w, -w], (np.r_[i, j, i, j], np.r_[i, j, j, i])), shape=(N, N)).tocsr()
    lu = splu((Lap[free][:, free] + prox * sp.eye(len(free))).tocsc())
    Lfc = Lap[free][:, np.where(fixed)[0]]
    e0 = Rest[i] - Rest[j]
    P = P_run.copy()
    Pfix = P[fixed]
    for it in range(iters):
        Sn = np.zeros((N, 3, 3))
        S = np.einsum('e,ea,eb->eab', w, e0, P[i] - P[j])
        np.add.at(Sn, i, S); np.add.at(Sn, j, S)
        U, _, Vt = np.linalg.svd(Sn)
        Vt[np.linalg.det(Vt.transpose(0, 2, 1) @ U.transpose(0, 2, 1)) < 0, 2] *= -1
        Rm = Vt.transpose(0, 2, 1) @ U.transpose(0, 2, 1)
        Re = 0.5 * (np.einsum('eab,eb->ea', Rm[i], e0) + np.einsum('eab,eb->ea', Rm[j], e0)) * w[:, None]
        rhs = np.zeros((N, 3)); np.add.at(rhs, i, Re); np.add.at(rhs, j, -Re)
        Pn = P.copy(); Pn[free] = lu.solve(rhs[free] - Lfc @ Pfix + prox * P[free])
        step = np.abs(Pn[free] - P[free]).max(); P = Pn
        if it % 10 == 0 or it == iters - 1: print(f'iter {it:2d}  max step {step:.3f} mm')
    return P


t0 = time.time()
OUT_ARAP = Path(str(OUT) + '_arap')
coords_arap = arap(coords_reg)
save_run(OUT_ARAP, coords_arap)
for f in ('coords_init.npy', 'coords_similarity.npy', 'run_meta.json', 'validation_summary.csv'):
    shutil.copy(OUT / f, OUT_ARAP)
print(f'ARAP {time.time() - t0:.0f}s; mean move on solid nodes {np.linalg.norm(coords_arap - coords_reg, axis=1)[np.unique(tmpl.cells_dict[12])].mean():.2f} mm')
df_int_arap = eval_interior(OUT_ARAP)
print(df_int_arap[df_int_arap.state == 'baseline registered'].to_string(index=False))
jac_arap = jacobian_check(OUT_ARAP)

## 10. Results table

In [ ]:
def results_table(case):
    rows = []
    for d in sorted(MAI.glob(f'out_{case}_*')):
        if not (d / 'interior_vs_gt.csv').exists() or not (d / 'jacobian' / 'jacobian_summary.json').exists(): continue
        v = pd.read_csv(d / 'validation_summary.csv').iloc[-1]
        meta = json.load(open(d / 'run_meta.json'))
        jj = json.load(open(d / 'jacobian' / 'jacobian_summary.json'))
        reg = pd.read_csv(d / 'interior_vs_gt.csv').query("state == 'baseline registered'").set_index('structure')
        rows.append(dict(run=d.name.replace(f'out_{case}_', ''), rigid_scale=meta['rigid_scale'], skin_mean=v['mean'], skin_hd=v['hausdorff'],
                         skull_out=reg.loc['skull_outer', 'mean_sym'], skull_in=reg.loc['skull_inner', 'mean_sym'],
                         brain=reg.loc['brain', 'mean_sym'], stem=reg.loc['brainstem', 'mean_sym'], eyes=reg.loc['eyes', 'mean_sym'],
                         brain_zext=reg.loc['brain', 'extent_ratio_z'], failing=jj['failing_registered'], newly_failing=jj['newly_failing']))
    return pd.DataFrame(rows).round(2)


res = results_table(CASE)
res.to_csv(MAI / f'results_{CASE}.csv', index=False)
res